# PDF Audiobook — optional Kokoro GPU run

This notebook keeps the desktop app unchanged. It clones the project, runs the existing headless pipeline with one Kokoro narrator on the selected Colab GPU, and downloads the verified M4B. Free Colab runtimes can disconnect or have no GPU; checkpoints live under `/content` unless you change the workspace path.

In [ ]:
import os, subprocess, sys, torch
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU is attached. Choose Runtime > Change runtime type > T4 GPU, then rerun.')
print(torch.cuda.get_device_name(0))

In [ ]:
# Linux tools used by the existing PDF parser and M4B finalizer.
!apt-get -qq update
!apt-get -qq install -y ffmpeg espeak-ng
!pip -q install uv

In [ ]:
# Colab's runtime Python may be 3.12; provision the project's pinned Python 3.11 environment.
REPO = '/content/AudiobookFree'
!rm -rf /content/AudiobookFree
!git clone --depth 1 https://github.com/AugustZhang1/AudiobookFree.git {REPO}
!uv python install 3.11
!uv venv --python 3.11 /content/audiobook-venv
!uv pip install --python /content/audiobook-venv/bin/python -e {REPO}
!uv pip install --python /content/audiobook-venv/bin/python 'kokoro==0.9.4' 'spacy<4' 'en_core_web_sm @ https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0.tar.gz'
PY = '/content/audiobook-venv/bin/python'

In [ ]:
from google.colab import files
uploaded = files.upload()
if not uploaded:
    raise RuntimeError('Upload one selectable-text PDF.')
pdf_name = next(iter(uploaded))
pdf_path = '/content/' + pdf_name
voice = 'af_heart' # @param ["af_heart", "af_alloy", "af_aoede", "af_bella", "af_jessica", "af_kore", "af_nicole", "af_nova", "af_river", "af_sarah", "af_sky", "am_adam", "am_echo", "am_eric", "am_fenrir", "am_liam", "am_michael", "am_onyx", "am_puck", "am_santa", "bf_alice", "bf_emma", "bf_isabella", "bf_lily", "bm_daniel", "bm_fable", "bm_george", "bm_lewis"]
speed = 1.0 # @param {type:"number", min:0.5, max:2.0, step:0.05}
chapter_mode = 'original' # @param ["original", "whole", "custom"]
chapter_count = 4 # @param {type:"integer", min:2, max:50}
workspace = '/content/pdf-audiobook-workspace'
output_dir = '/content/pdf-audiobook-output'

In [ ]:
# One long, resumable headless run. Rerun this cell after a disconnect with the same PDF/settings.
if chapter_mode != 'custom':
    chapter_count = None
cmd = [PY, '-u', '-m', 'pdf_audiobook.colab', pdf_path, '--workspace-root', workspace, '--output-dir', output_dir, '--voice', voice, '--speed', str(speed), '--chapter-mode', chapter_mode]
if chapter_count is not None:
    cmd += ['--chapter-count', str(chapter_count)]
process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
verified_path = None
assert process.stdout is not None
for line in process.stdout:
    print(line, end='')
    if line.startswith('Verified M4B: '):
        verified_path = line.removeprefix('Verified M4B: ').strip()
return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f'Colab conversion failed with exit code {return_code}')
if not verified_path:
    raise RuntimeError('The runner did not report a verified M4B path.')
files.download(verified_path)